## Библиотеки

In [ ]:
!pip install ase mace-torch mplcursors ipympl

In [1]:
%matplotlib widget

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import ase
from mace.calculators import MACECalculator
import os
import torch
import umap
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN, AgglomerativeClustering, SpectralClustering
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
from IPython.display import display
import ipywidgets as widgets
import matplotlib as mpl
from ase.visualize.plot import plot_atoms
from configuration import Configuration

## Инициализация датасета 

Сейчас мы смотрим только на каждый 20-й .xyz датасета, для этих же .xyz и были посчитаны фингерпринты. Чтобы это поменять (т.е. обсчитывать больше конфигураций) нужно поменять переменную trim_factor - сейчас он 20.

In [58]:
trim_factor = 3

In [59]:
data_cfg = Configuration.from_file("polymers.cfg")[::trim_factor]

In [60]:
type_map = {0: 'C', 1: 'H', 2: 'O'}
data = [i.to_ase(type_map) for i in data_cfg]

/media/daniil/Kingston/bmstu_polymers/rep/bmstu_polymers/configuration.py:217: FutureWarning: Please use atoms.calc = calc
  atoms.set_calculator(calculator)


In [61]:
len(data)

6566

### Расчёт фингерпринтов (скорее всего не надо)

Запускать, **только** если нужно заново посчитать фингерпринты (например, при смене trim_factor или другом датасете)

In [66]:
if torch.cuda.is_available():
  device='cuda'
else:
  device='cpu'
calculator = MACECalculator(
    model_paths="2023-12-03-mace-mp.model",
    device=device
)

/media/daniil/Kingston/miniconda3/lib/python3.11/site-packages/mace/calculators/mace.py:143: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Using head Default out of ['Default']
No dtype selected, switching to float64 to match model dtype.


In [ ]:
for i, atoms in enumerate(data[669:]):
    if i % 10 == 0:
        print(i)
        if i % 100 == 0:
          torch.cuda.empty_cache()
    fps.append(calculator.get_descriptors(atoms))

In [ ]:
fps_tot = np.concatenate(fps, axis=0)
np.save("fps_tot", fps_tot)

In [ ]:
for i, fp in enumerate(fps):
  np.save(f"fps/fp_{i}", fp)

In [ ]:
torch.cuda.empty_cache()

In [ ]:
fps_tot_scaler = StandardScaler().fit(fps_tot)
fps_tot_scaled = fps_tot_scaler.transform(fps_tot)
fps_scaled = [fps_tot_scaler.transform(fp) for fp in fps]

### Загрузка существующих фингерпринтов

Если ничего не меняли, то можно скачать уже посчитанные фингерпринты.

In [62]:
fps_tot = np.load("fps_big.npy")
fps = []
#print(fps_tot[-1, -1])
for i in range(0, int(fps_tot[-1, -1]), trim_factor):
    if i % 100 == 0:
        print(i)
    chosen = fps_tot[:, -1] == i
    #print(len(chosen))
    fps.append(fps_tot[chosen, :-1])
fps_tot = fps_tot[:, :-1]

0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300
3600
3900
4200
4500
4800
5100
5400
5700
6000
6300
6600
6900
7200
7500
7800
8100
8400
8700
9000
9300
9600
9900
10200
10500
10800
11100
11400
11700
12000
12300
12600
12900
13200
13500
13800
14100
14400
14700
15000
15300
15600
15900
16200
16500
16800
17100
17400
17700
18000
18300
18600
18900
19200
19500


In [63]:
fps_tot_scaler = StandardScaler().fit(fps_tot)
fps_tot_scaled = fps_tot_scaler.transform(fps_tot)
fps_scaled = [fps_tot_scaler.transform(fp) for fp in fps]

# Вариант 1
Кластеризация по отдельным фингерпринтам

Работает ОЧЕНЬ долго. Не получилось

In [ ]:
# fps_scaled = StandardScaler().fit_transform(fps_tot)

In [ ]:
# reducer = umap.UMAP(min_dist=0.003, n_neighbors=30)

In [ ]:
# embedding = reducer.fit_transform(fps_scaled)
# embedding.shape

In [ ]:
#clusterizer = DBSCAN(eps=0.05).fit(fps_scaled)
# clusterizer = AgglomerativeClustering(n_clusters=None, distance_threshold=0.5).fit(fps_scaled)

In [ ]:
# plt.figure(dpi=500)
# plt.scatter(
#     embedding[:, 0],
#     embedding[:, 1],
#     c=clusterizer.labels_
# )
# plt.gca().set_aspect('equal', 'datalim')
# plt.title('UMAP projection of the allotropes dataset', fontsize=24);

In [ ]:
# np.sum(clusterizer.labels_ == -1)

# Вариант 2
Предварительная кластеризация  внутри конфигураций.

Этот вариант получился.

Параметры:

$n_{clusters}$ - столько характерных фингерпринтов мы оставляем для **каждой** конфигурации (т.е. превращаем массив поатомных фингерпринтов в n фингерпринтов). Чем больше, тем больше информации мы сохраняем о каждой конфигурации, но теряем в скорости и (возможно) интерпретируемости конечной диаграммы.


In [64]:
n_clusters = 3

### Считаем кластеризацию

In [65]:
fp_centroids = np.zeros((len(fps_scaled), n_clusters, 256))
for i, fp in enumerate(fps_scaled):
  was_err = False
  try:
    clusterizer = AgglomerativeClustering(n_clusters=n_clusters).fit(fp)
  except ValueError:
    clusterizer = AgglomerativeClustering(n_clusters=1).fit(fp)
  fp_clustered = [fp[clusterizer.labels_ == label] for label in np.unique(clusterizer.labels_)]
  #print([np.mean(cl, axis=0) for cl in fp_clustered])
  fp_centroids[i] = np.array([np.mean(cl, axis=0) for cl in fp_clustered])
fp_centroids = fp_centroids.reshape((-1, fp_centroids.shape[-1]))
fp_centroids.shape

(19698, 256)

**Уменьшаем размерность для диаграммы** (до двух компонент)

In [66]:
reducer = umap.UMAP(
    min_dist = 0.1,
    n_neighbors = 40
)
embedding = reducer.fit_transform(fp_centroids)
embedding.shape

(19698, 2)

**Уменьшаем размерность для кластеризации** (до $n_{components}$ компонент)

In [67]:
reducer_cl = umap.UMAP(
    min_dist = 0.1,
    n_components = 30,
    n_neighbors = 15
)
embedding_cl = reducer_cl.fit_transform(fp_centroids)

Кластеризуем:

In [ ]:
clusterizer = AgglomerativeClustering(n_clusters=None, distance_threshold=20.0).fit(embedding_cl)

In [ ]:
np.save("2d_embedding", embedding)
np.save("clusterization_embedding", embedding_cl)
np.save("labels", clusterizer.labels_)
labels = clusterizer.labels_

### Скачиваем кластеризацию

In [66]:
embedding = np.load("2d_embedding.npy")
embedding_cl = np.load("clusterization_embedding.npy")
labels = np.load("labels.npy")

## Визуализация

In [ ]:
%matplotlib widget

In [ ]:
input_exists = False

In [ ]:
max_cluster = np.max(labels)
cmap = mpl.colormaps['viridis']
cmap_highlighted = mpl.colormaps['hsv']

# Генерация случайных данных
np.random.seed(42)
n_points = len(fps)
x = embedding[:, 0]
y = embedding[:, 1]

# Функция для генерации изображения
def render_snapshot(i):
    fig, ax = plt.subplots(figsize=(5, 5))
    plot_atoms(data[i], ax, radii=0.3, rotation=('-10x,20y,-1z'))
    #ax.imshow(np.random.rand(10, 10), cmap='viridis')
    #ax.set_title(f'Image for point {i}')
    plt.axis('off')
    plt.close()
    return fig

# Создаем основную фигуру
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Инициализируем массив цветов
colors=cmap(labels * (1.0 / max_cluster))

# Рисуем точки на левом графике
scatter = ax1.scatter(x, y, c=colors, s=40, picker=True, pickradius=5)
highlight_points = [0] * n_clusters
for j in range(n_clusters):
    highlight_points[j] = ax1.scatter([], [], c=[cmap_highlighted(j * (1.0 / n_clusters))], s=120, edgecolors='black', linewidths=1.5)

if input_exists:
    plt.scatter(
        input_embedding[:, 0],
        input_embedding[:, 1],
        c='orange'
    )
    
ax1.set_title('Точки данных')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')

# Настраиваем правый график
ax2.axis('off')
ax2.set_title('Изображение будет здесь')

last_index = None
last_color = None

# Функция для обновления графика
def update_image(index):
    for j in range(n_clusters):
        highlight_points[j].set_offsets([[x[n_clusters*index+j], y[n_clusters*index+j]]])
        #print(n_clusters*index+j)
    print(20*j)
    
    # Обновляем цвета точек
    # global colors
    # global last_index
    # global last_color
    # if last_index is not None:
    #    colors[last_index] = last_color
    # last_color = colors[index]
    # last_index = index
    # colors[index] = (1.0, 0.0, 0.0, 1.0)
    # colors = ['blue'] * n_points  # Все точки синие
    # colors[index] = 'red'         # Выбранная точка - красная
    
    # Обновляем scatter plot
    scatter.set_facecolors(colors)
    
    # Обновляем изображение
    ax2.clear()
    ax2.axis('off')
    img_fig = render_snapshot(index)
    img_fig.canvas.draw()
    img_array = np.array(img_fig.canvas.renderer.buffer_rgba())
    ax2.imshow(img_array)
    ax2.set_title(f'Image for point {index*trim_factor} in dataset', fontsize=12)
    
    # Перерисовываем фигуру
    fig.canvas.draw_idle()

# Создаем интерактивный элемент
interact(update_image, index=IntSlider(
    min=0, 
    max=n_points-1, 
    step=1, 
    value=0,
    description='Point index:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
))

update_image(0)

plt.tight_layout()
plt.show()

925